# LeetCode 1275 — Find Winner on a Tic Tac Toe Game bằng Python

**Link đề:** https://leetcode.com/problems/find-winner-on-a-tic-tac-toe-game/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Matrix, Simulation

```python
class Solution:
    def tictactoe(self, moves: list[list[int]]) -> str:
        ...
```

## Mục tiêu học
1. Mô phỏng bàn cờ `3 x 3` bằng **list 2 chiều**, biết lượt của ai bằng `i % 2`.
2. Liệt kê đủ **8 đường thắng** (3 hàng, 3 cột, 2 đường chéo) và kiểm tra bằng code gọn: `all`, `zip(*board)`.
3. Tối ưu thành **bộ đếm theo hàng/cột/chéo** (`+1` cho A, `-1` cho B) — chỉ `O(1)` cho mỗi nước đi, tổng quát được cho bàn `n x n`.
4. Dùng `set` các ô + danh sách đường thắng để kiểm tra kiểu "tập con".
5. Hiểu thứ tự kết luận: **thắng → hòa (đủ 9 nước) → còn chơi (Pending)**; trace, test, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và ví dụ

Hai người A và B chơi cờ ca-rô (tic-tac-toe) trên lưới `3 x 3`:
- A luôn đi trước và đặt `'X'`; B đặt `'O'`. Hai người đi luân phiên.
- Chỉ đặt vào ô trống. Trò chơi kết thúc khi có **3 ký tự giống nhau** trên một hàng, một cột hoặc một đường chéo; hoặc khi lưới đầy.
- Sau khi kết thúc không còn nước đi nào.

Cho `moves`, với `moves[i] = [row, col]` là nước thứ `i`. Trả về `"A"` hoặc `"B"` nếu có người thắng; `"Draw"` nếu hòa (lưới đầy); `"Pending"` nếu ván chưa xong.

Đề đảm bảo `moves` **hợp lệ** (đúng luật, không trùng ô, A đi trước).

| `moves` | Output |
|---|---|
| `[[0,0],[2,0],[1,1],[2,1],[2,2]]` | `"A"` |
| `[[0,0],[1,1],[0,1],[0,2],[1,0],[2,0]]` | `"B"` |
| `[[0,0],[1,1],[2,0],[1,0],[1,2],[2,1],[0,1],[0,2],[2,2]]` | `"Draw"` |

**Constraints:** `1 <= moves.length <= 9`, `0 <= row, col <= 2`, không lặp nước đi.

### Vẽ tay ví dụ 1 và 2

```text
Ví dụ 1 (A thắng đường chéo chính)    Ví dụ 2 (B thắng đường chéo phụ)
 X | . | .                              X | X | O
 . | X | .                              X | O | .
 O | O | X                              O | . | .
```

Nước chẵn (`i = 0, 2, 4, …`) là của A, nước lẻ là của B.

In [1]:
def draw_board(moves):
    board = [[" "] * 3 for _ in range(3)]
    for i, (r, c) in enumerate(moves):
        board[r][c] = "X" if i % 2 == 0 else "O"
    for r in range(3):
        print(" " + " | ".join(ch if ch != " " else "." for ch in board[r]))
    return board

draw_board([[0, 0], [2, 0], [1, 1], [2, 1], [2, 2]])
print()
draw_board([[0, 0], [1, 1], [0, 1], [0, 2], [1, 0], [2, 0]])

 X | . | .
 . | X | .
 O | O | X

 X | X | O
 X | O | .
 O | . | .


[['X', 'X', 'O'], ['X', 'O', ' '], ['O', ' ', ' ']]

## 2. Nền tảng Python cần biết

### 2.1. Lượt của ai: `i % 2` và `enumerate`

`enumerate(moves)` cho cặp `(i, move)`. `i % 2 == 0` → A (X), ngược lại B (O).
Tách 2 tầng: `for i, (r, c) in enumerate(moves)`.

In [2]:
moves = [[0, 0], [2, 0], [1, 1]]
for i, (r, c) in enumerate(moves):
    who = "A" if i % 2 == 0 else "B"
    print(f"nước {i}: {who} đánh ({r}, {c})")

nước 0: A đánh (0, 0)
nước 1: B đánh (2, 0)
nước 2: A đánh (1, 1)


### 2.2. Tách nước của A và B bằng slicing bước 2

`moves[0::2]` lấy phần tử ở vị trí 0, 2, 4, … (của A); `moves[1::2]` lấy 1, 3, 5, … (của B).
Cú pháp `a[start:stop:step]`, bỏ trống `stop` = tới hết.

In [3]:
moves = [[0, 0], [2, 0], [1, 1], [2, 1], [2, 2]]
print("A:", moves[0::2])   # [[0,0],[1,1],[2,2]]
print("B:", moves[1::2])   # [[2,0],[2,1]]

A: [[0, 0], [1, 1], [2, 2]]
B: [[2, 0], [2, 1]]


### 2.3. `all(...)` — mọi phần tử đều thỏa

`all(board[r][0] == "X" for r in range(3))` đúng khi cả 3 ô của cột 0 là X.

In [4]:
board = [["X", "O", " "], ["X", " ", " "], ["X", " ", "O"]]
print(all(board[r][0] == "X" for r in range(3)))   # True: cột 0 toàn X
print(all(board[0][c] == "X" for c in range(3)))   # False: hàng 0 không toàn X

True
False


### 2.4. Lấy các cột bằng `zip(*board)`

`*board` "rải" 3 hàng thành 3 đối số của `zip`. `zip(h0, h1, h2)` ghép phần tử cùng vị trí → mỗi bộ là **một cột**. Đây là cách **chuyển vị (transpose)** nhanh.

In [5]:
board = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print(list(zip(*board)))     # [(1, 4, 7), (2, 5, 8), (3, 6, 9)]
print([board[i][i] for i in range(3)])       # đường chéo chính: [1, 5, 9]
print([board[i][2 - i] for i in range(3)])   # đường chéo phụ:  [3, 5, 7]

[(1, 4, 7), (2, 5, 8), (3, 6, 9)]
[1, 5, 9]
[3, 5, 7]


### 2.5. `set` của tuple và kiểm tra tập con

List không cho vào `set` được (list **thay đổi được** nên không "băm" — hash — được), nhưng **tuple** thì được. `{(0,0), (1,1), (2,2)} <= S` hỏi: cả 3 ô có nằm trong `S` không?

In [6]:
a_cells = {tuple(m) for m in [[0, 0], [1, 1], [2, 2]]}
print(a_cells)
diag = {(0, 0), (1, 1), (2, 2)}
print(diag <= a_cells)                        # True
print({(0, 0), (0, 1), (0, 2)} <= a_cells)    # False
try:
    {[0, 0]}
except TypeError as e:
    print("TypeError:", e)

{(1, 1), (2, 2), (0, 0)}
True
False
TypeError: unhashable type: 'list'


## 3. Cách 1 — Dựng bàn cờ rồi kiểm tra 8 đường

1. Đặt tất cả nước đi lên `board`.
2. Với mỗi người chơi (`"X"` → `"A"`, `"O"` → `"B"`), kiểm tra 3 hàng, 3 cột, 2 chéo.
3. Không ai thắng: `len(moves) == 9` → `"Draw"`, ngược lại `"Pending"`.

In [7]:
def wins(board, ch):
    for i in range(3):
        if all(board[i][j] == ch for j in range(3)):    # hàng i
            return True
        if all(board[j][i] == ch for j in range(3)):    # cột i
            return True
    if all(board[i][i] == ch for i in range(3)):        # chéo chính
        return True
    if all(board[i][2 - i] == ch for i in range(3)):    # chéo phụ
        return True
    return False


def tictactoe_board(moves):
    board = [[" "] * 3 for _ in range(3)]
    for i, (r, c) in enumerate(moves):
        board[r][c] = "X" if i % 2 == 0 else "O"
    if wins(board, "X"):
        return "A"
    if wins(board, "O"):
        return "B"
    return "Draw" if len(moves) == 9 else "Pending"

print(tictactoe_board([[0, 0], [2, 0], [1, 1], [2, 1], [2, 2]]))                     # A
print(tictactoe_board([[0, 0], [1, 1], [0, 1], [0, 2], [1, 0], [2, 0]]))             # B
print(tictactoe_board([[0, 0], [1, 1], [2, 0], [1, 0], [1, 2], [2, 1], [0, 1], [0, 2], [2, 2]]))  # Draw
print(tictactoe_board([[0, 0], [1, 1]]))                                             # Pending

A
B
Draw
Pending


### Viết gọn việc liệt kê 8 đường

Gom 8 đường vào một list rồi dùng `any(...)`. Cách này dễ đọc và ít quên đường nào.

In [8]:
def lines_of(board):
    rows = [list(r) for r in board]
    cols = [list(c) for c in zip(*board)]
    diags = [[board[i][i] for i in range(3)], [board[i][2 - i] for i in range(3)]]
    return rows + cols + diags


def tictactoe_lines(moves):
    board = [[" "] * 3 for _ in range(3)]
    for i, (r, c) in enumerate(moves):
        board[r][c] = "X" if i % 2 == 0 else "O"
    for ch, name in (("X", "A"), ("O", "B")):
        if any(line == [ch, ch, ch] for line in lines_of(board)):
            return name
    return "Draw" if len(moves) == 9 else "Pending"

print(len(lines_of([[" "] * 3 for _ in range(3)])))       # 8 đường
print(tictactoe_lines([[0, 0], [1, 1], [0, 1], [0, 2], [1, 0], [2, 0]]))   # B

8
B


**Độ phức tạp Cách 1:** bàn cố định `3 x 3` nên mọi thứ là hằng số, `O(1)`. Nhưng nếu tổng quát lên bàn `n x n`, kiểm tra mọi đường tốn `O(n²)` — và nếu phải kiểm tra **sau mỗi nước** thì `O(n²)` mỗi nước.

## 4. Nút thắt & insight

1. **Chỉ người đi nước cuối mới có thể thắng.** Vì `moves` hợp lệ, trò chơi dừng ngay khi có người thắng → nếu có người thắng, chính họ đã đi nước cuối. Không cần kiểm tra người kia.
2. **Không cần nhìn lại cả bàn:** một nước ở `(r, c)` chỉ ảnh hưởng hàng `r`, cột `c`, và (có thể) 2 đường chéo. Ta có thể **đếm** số quân của mỗi người trên từng đường.
3. **Gộp 2 người vào 1 bộ đếm:** cộng `+1` cho A, `-1` cho B. Một đường có giá trị `+3` ⇔ A chiếm cả 3 ô; `-3` ⇔ B chiếm cả 3 ô (vì mỗi đường chỉ có 3 ô).

In [9]:
# Minh họa bộ đếm +1/-1 cho hàng 0
row0 = 0
for who in ["A", "B", "A"]:
    row0 += 1 if who == "A" else -1
    print(who, "-> row0 =", row0)
print("hàng 0 bị trộn, không ai thắng (|row0| != 3)")
row0 = 3
print("nếu row0 == 3 -> A chiếm đủ 3 ô")

A -> row0 = 1
B -> row0 = 0
A -> row0 = 1
hàng 0 bị trộn, không ai thắng (|row0| != 3)
nếu row0 == 3 -> A chiếm đủ 3 ô


## 5. Cách 2 — Bộ đếm hàng/cột/chéo (lời giải chính)

Duy trì:
- `rows[3]`, `cols[3]`: tổng có dấu trên từng hàng/cột.
- `diag`: tổng trên đường chéo chính (ô có `r == c`).
- `anti`: tổng trên đường chéo phụ (ô có `r + c == 2`).

Mỗi nước: cộng `v = +1` (A) hoặc `-1` (B) vào các bộ đếm liên quan, rồi kiểm tra có bộ nào đạt `|3|` không.

In [10]:
def tictactoe_counter(moves):
    n = 3
    rows = [0] * n
    cols = [0] * n
    diag = anti = 0
    for i, (r, c) in enumerate(moves):
        v = 1 if i % 2 == 0 else -1
        rows[r] += v
        cols[c] += v
        if r == c:
            diag += v
        if r + c == n - 1:
            anti += v
        if abs(rows[r]) == n or abs(cols[c]) == n or abs(diag) == n or abs(anti) == n:
            return "A" if v == 1 else "B"
    return "Draw" if len(moves) == n * n else "Pending"

print(tictactoe_counter([[0, 0], [2, 0], [1, 1], [2, 1], [2, 2]]))               # A
print(tictactoe_counter([[0, 0], [1, 1], [0, 1], [0, 2], [1, 0], [2, 0]]))       # B
print(tictactoe_counter([[0, 0], [1, 1], [2, 0], [1, 0], [1, 2], [2, 1], [0, 1], [0, 2], [2, 2]]))  # Draw

A
B
Draw


### Vì sao `r == c` và `r + c == 2`?

| ô | `r == c`? | `r + c == 2`? |
|---|---|---|
| (0,0) | ✔ | |
| (1,1) | ✔ | ✔ (ô trung tâm thuộc **cả hai** chéo) |
| (2,2) | ✔ | |
| (0,2) | | ✔ |
| (2,0) | | ✔ |

Dùng hai `if` **riêng** (không phải `elif`) để ô trung tâm được cộng vào cả hai đường chéo.

In [11]:
for r in range(3):
    print(" ".join(("D" if r == c else ".") + ("A" if r + c == 2 else ".") for c in range(3)))
# D = chéo chính, A = chéo phụ; ô giữa in "DA"

D. .. .A
.. DA ..
.A .. D.


## 6. Cách 3 — Tập ô của người đi cuối + danh sách 8 đường thắng

Dùng insight 1: chỉ cần xét người đi **nước cuối**. Lấy tập ô của họ (`moves[-1::-2]` — đi lùi từ nước cuối, bước 2), rồi kiểm tra có đường thắng nào là **tập con**.

In [12]:
WIN_LINES = (
    [{(r, c) for c in range(3)} for r in range(3)]          # 3 hàng
    + [{(r, c) for r in range(3)} for c in range(3)]        # 3 cột
    + [{(i, i) for i in range(3)}, {(i, 2 - i) for i in range(3)}]   # 2 chéo
)
print(len(WIN_LINES), WIN_LINES[-1])

def tictactoe_sets(moves):
    last_player = "A" if len(moves) % 2 == 1 else "B"
    cells = {tuple(m) for m in moves[-1::-2]}       # các nước của người đi cuối
    if any(line <= cells for line in WIN_LINES):
        return last_player
    return "Draw" if len(moves) == 9 else "Pending"

moves = [[0, 0], [1, 1], [0, 1], [0, 2], [1, 0], [2, 0]]
print(moves[-1::-2])            # nước 5, 3, 1 -> của B
print(tictactoe_sets(moves))    # B

8 {(1, 1), (0, 2), (2, 0)}
[[2, 0], [0, 2], [1, 1]]
B


`len(moves) % 2 == 1` → số nước lẻ → nước cuối có chỉ số chẵn → của A.

## 7. Cách 4 — Mã hóa bitmask (nâng cao, tham khảo)

Đánh số ô `(r, c)` thành bit `3·r + c` (0..8). Tập ô của một người là một số nguyên 9 bit. Đường thắng cũng là số 9 bit; người đó thắng nếu `mask & line == line`.

In [13]:
WIN_MASKS = [sum(1 << (3 * r + c) for (r, c) in line) for line in WIN_LINES]
print([bin(m) for m in WIN_MASKS[:3]])   # 3 hàng: 0b111, 0b111000, 0b111000000

def tictactoe_bitmask(moves):
    masks = [0, 0]
    for i, (r, c) in enumerate(moves):
        masks[i % 2] |= 1 << (3 * r + c)
    for p, name in ((0, "A"), (1, "B")):
        if any(masks[p] & w == w for w in WIN_MASKS):
            return name
    return "Draw" if len(moves) == 9 else "Pending"

print(tictactoe_bitmask([[0, 0], [2, 0], [1, 1], [2, 1], [2, 2]]))   # A

['0b111', '0b111000', '0b111000000']
A


- `1 << k` = số có duy nhất bit thứ `k` bật (= `2**k`).
- `x |= y` bật thêm các bit của `y` vào `x`.
- Trong Python, `&` có độ ưu tiên **cao hơn** phép so sánh `==`, nên `masks[p] & w == w` được hiểu là `(masks[p] & w) == w` — đúng ý ta. (Khác với C/Java, nơi `==` được ưu tiên trước `&`!) Ghi ngoặc rõ ràng vẫn là thói quen tốt.

In [14]:
print(1 << 0, 1 << 3, 1 << 8)       # 1 8 256
x = 0
x |= 1 << 0
x |= 1 << 4
print(bin(x))                       # 0b10001
print((0b111 & 0b10111) == 0b111)   # True

1 8 256
0b10001
True


## 8. Trace từng bước (Cách 2)

Ví dụ 2: `[[0,0],[1,1],[0,1],[0,2],[1,0],[2,0]]`.

| i | người | (r,c) | v | rows | cols | diag | anti | ai đạt 3? |
|---:|---|---|---:|---|---|---:|---:|---|
| 0 | A | (0,0) | +1 | [1,0,0] | [1,0,0] | 1 | 0 | |
| 1 | B | (1,1) | −1 | [1,−1,0] | [1,−1,0] | 0 | −1 | |
| 2 | A | (0,1) | +1 | [2,−1,0] | [1,0,0] | 0 | −1 | |
| 3 | B | (0,2) | −1 | [1,−1,0] | [1,0,−1] | 0 | −2 | |
| 4 | A | (1,0) | +1 | [1,0,0] | [2,0,−1] | 0 | −2 | |
| 5 | B | (2,0) | −1 | [1,0,−1] | [1,0,−1] | 0 | **−3** | B thắng |

In [15]:
def trace_tictactoe(moves):
    rows, cols = [0] * 3, [0] * 3
    diag = anti = 0
    for i, (r, c) in enumerate(moves):
        v = 1 if i % 2 == 0 else -1
        rows[r] += v
        cols[c] += v
        if r == c:
            diag += v
        if r + c == 2:
            anti += v
        print(f"i={i} {'A' if v == 1 else 'B'} ({r},{c}) v={v:+d} | rows={rows} cols={cols} diag={diag:+d} anti={anti:+d}")
        if 3 in (abs(rows[r]), abs(cols[c]), abs(diag), abs(anti)):
            print("  -> có đường đạt |3|, người vừa đi thắng")
            return "A" if v == 1 else "B"
    res = "Draw" if len(moves) == 9 else "Pending"
    print("  -> không ai thắng:", res)
    return res

trace_tictactoe([[0, 0], [1, 1], [0, 1], [0, 2], [1, 0], [2, 0]])

i=0 A (0,0) v=+1 | rows=[1, 0, 0] cols=[1, 0, 0] diag=+1 anti=+0
i=1 B (1,1) v=-1 | rows=[1, -1, 0] cols=[1, -1, 0] diag=+0 anti=-1
i=2 A (0,1) v=+1 | rows=[2, -1, 0] cols=[1, 0, 0] diag=+0 anti=-1
i=3 B (0,2) v=-1 | rows=[1, -1, 0] cols=[1, 0, -1] diag=+0 anti=-2
i=4 A (1,0) v=+1 | rows=[1, 0, 0] cols=[2, 0, -1] diag=+0 anti=-2
i=5 B (2,0) v=-1 | rows=[1, 0, -1] cols=[1, 0, -1] diag=+0 anti=-3
  -> có đường đạt |3|, người vừa đi thắng


'B'

`f"{v:+d}"` in số nguyên **kèm dấu** (`+1`, `-1`) — giúp bảng trace dễ đọc.

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def tictactoe(self, moves: list[list[int]]) -> str:
        n = 3
        rows = [0] * n
        cols = [0] * n
        diag = anti = 0
        for i, (r, c) in enumerate(moves):
            v = 1 if i % 2 == 0 else -1
            rows[r] += v
            cols[c] += v
            if r == c:
                diag += v
            if r + c == n - 1:
                anti += v
            if n in (abs(rows[r]), abs(cols[c]), abs(diag), abs(anti)):
                return "A" if v == 1 else "B"
        return "Draw" if len(moves) == n * n else "Pending"

print(Solution().tictactoe([[0, 0], [2, 0], [1, 1], [2, 1], [2, 2]]))   # A

A


## 10. Vì sao đúng?

- **Bộ đếm phản ánh chính xác:** invariant sau mỗi nước — `rows[r]` = (số X trên hàng r) − (số O trên hàng r); tương tự cột, chéo.
- **`|tổng| = 3` ⇔ một người chiếm cả 3 ô:** mỗi đường có đúng 3 ô, mỗi ô đóng góp `+1`, `-1` hoặc `0`. Tổng bằng `3` chỉ khi cả 3 ô là `+1` (X); `-3` chỉ khi cả 3 là `-1` (O).
- **Chỉ kiểm tra đường đi qua ô vừa đánh là đủ:** một đường chỉ thay đổi khi có quân mới đặt lên nó; nếu nó chưa đạt `|3|` trước nước này thì chỉ có thể đạt sau khi được cộng thêm.
- **Thứ tự kết luận:** nếu không có đường nào đạt 3 sau mọi nước → không ai thắng. Khi đó lưới đầy (`9` nước) ⇒ `"Draw"`, chưa đầy ⇒ `"Pending"`.

## 11. Độ phức tạp và so sánh

Gọi `k = len(moves)` (≤ 9), tổng quát bàn `n x n`.

| Cách | Thời gian (bàn n×n) | Bộ nhớ | Ghi chú |
|---|---|---|---|
| 1. Dựng bàn + 8 đường | `O(k + n²)` | `O(n²)` | trực quan nhất |
| 2. Bộ đếm ±1 | `O(k)` | `O(n)` | lời giải chính, tổng quát tốt |
| 3. Set + đường thắng | `O(k + n²)` | `O(n²)` | gọn, dùng insight "người đi cuối" |
| 4. Bitmask | `O(k + số đường)` | `O(1)` (bàn 3x3) | đẹp cho bàn nhỏ |

Với bàn `3 x 3` cố định, tất cả đều `O(1)`.

## 12. Bộ test

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.tictactoe([[0, 0], [2, 0], [1, 1], [2, 1], [2, 2]]), "A")
check("đề 2", sol.tictactoe([[0, 0], [1, 1], [0, 1], [0, 2], [1, 0], [2, 0]]), "B")
check("đề 3", sol.tictactoe([[0, 0], [1, 1], [2, 0], [1, 0], [1, 2], [2, 1], [0, 1], [0, 2], [2, 2]]), "Draw")
check("1 nước", sol.tictactoe([[1, 1]]), "Pending")
check("A thắng hàng", sol.tictactoe([[0, 0], [1, 0], [0, 1], [1, 1], [0, 2]]), "A")
check("A thắng cột", sol.tictactoe([[0, 1], [0, 0], [1, 1], [0, 2], [2, 1]]), "A")
check("A thắng chéo phụ", sol.tictactoe([[0, 2], [0, 0], [1, 1], [0, 1], [2, 0]]), "A")
check("A thắng ở nước thứ 9", sol.tictactoe([[0, 0], [0, 1], [0, 2], [1, 0], [1, 1], [1, 2], [2, 1], [2, 0], [2, 2]]), "A")
check("8 nước chưa ai thắng", sol.tictactoe([[0, 0], [1, 1], [2, 0], [1, 0], [1, 2], [2, 1], [0, 1], [0, 2]]), "Pending")

PASS đề 1: A
PASS đề 2: B
PASS đề 3: Draw
PASS 1 nước: Pending
PASS A thắng hàng: A
PASS A thắng cột: A
PASS A thắng chéo phụ: A
PASS A thắng ở nước thứ 9: A
PASS 8 nước chưa ai thắng: Pending


### Đối chiếu trên **mọi ván hợp lệ** có thể xảy ra

Ta sinh tất cả dãy nước đi hợp lệ bằng **đệ quy** (backtracking): thử đặt vào mỗi ô trống, dừng nhánh khi có người thắng. Mọi tiền tố (prefix) của một ván hợp lệ cũng là input hợp lệ. Có hơn nửa triệu ván, ta lấy mẫu để chạy nhanh.

In [18]:
all_inputs = []

def gen(moves, used):
    all_inputs.append([m[:] for m in moves])
    if tictactoe_board(moves) in ("A", "B") or len(moves) == 9:
        return                          # ván đã kết thúc, không đi tiếp
    for r in range(3):
        for c in range(3):
            if (r, c) not in used:
                used.add((r, c))
                moves.append([r, c])
                gen(moves, used)
                moves.pop()
                used.remove((r, c))

for r in range(3):
    for c in range(3):
        gen([[r, c]], {(r, c)})
print("số input hợp lệ:", len(all_inputs))

import random
random.seed(1275)
sample = random.sample(all_inputs, 20000)
methods = [tictactoe_board, tictactoe_lines, tictactoe_counter, tictactoe_sets, tictactoe_bitmask]
for mv in sample:
    expected = tictactoe_board(mv)
    for f in methods:
        assert f(mv) == expected, (f.__name__, mv)
    assert Solution().tictactoe(mv) == expected
from collections import Counter
print("20000 mẫu: 5 cách + Solution khớp. Phân bố:", Counter(tictactoe_board(mv) for mv in sample))

số input hợp lệ: 549945


20000 mẫu: 5 cách + Solution khớp. Phân bố: Counter({'Pending': 10834, 'A': 4687, 'B': 2830, 'Draw': 1649})


`Counter` (từ `collections`) đếm số lần xuất hiện của mỗi giá trị — tiện để xem test có đủ cả 4 loại kết quả.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Kết luận "Draw" khi chưa đủ 9 nước

Không ai thắng **chưa chắc** là hòa: nếu còn ô trống thì là `"Pending"`.

In [19]:
def wrong_draw(moves):
    res = tictactoe_board(moves)
    return "Draw" if res in ("Draw", "Pending") else res     # SAI

print("sai :", wrong_draw([[0, 0], [1, 1]]), "| đúng:", Solution().tictactoe([[0, 0], [1, 1]]))

sai : Draw | đúng: Pending


### Lỗi 2: Kiểm tra Draw **trước** kiểm tra thắng

Ván kết thúc ở nước thứ 9 bằng một chiến thắng (xem test "A thắng ở nước thứ 9") sẽ bị báo nhầm là `"Draw"`.

In [20]:
def wrong_order(moves):
    if len(moves) == 9:
        return "Draw"                     # SAI: chưa xét thắng
    return tictactoe_counter(moves)

mv = [[0, 0], [0, 1], [0, 2], [1, 0], [1, 1], [1, 2], [2, 1], [2, 0], [2, 2]]
print("sai :", wrong_order(mv), "| đúng:", Solution().tictactoe(mv))

sai : Draw | đúng: A


### Lỗi 3: Dùng `elif` cho hai đường chéo

Ô trung tâm `(1,1)` thuộc **cả hai** chéo. Dùng `elif` thì chéo phụ không được cộng.

In [21]:
def wrong_elif(moves):
    rows, cols = [0] * 3, [0] * 3
    diag = anti = 0
    for i, (r, c) in enumerate(moves):
        v = 1 if i % 2 == 0 else -1
        rows[r] += v; cols[c] += v
        if r == c:
            diag += v
        elif r + c == 2:                  # SAI
            anti += v
        if 3 in (abs(rows[r]), abs(cols[c]), abs(diag), abs(anti)):
            return "A" if v == 1 else "B"
    return "Draw" if len(moves) == 9 else "Pending"

mv = [[0, 2], [0, 0], [1, 1], [0, 1], [2, 0]]   # A thắng chéo phụ qua tâm
print("sai :", wrong_elif(mv), "| đúng:", Solution().tictactoe(mv))

sai : Pending | đúng: A


### Lỗi 4: Đếm chung không có dấu (cộng +1 cho cả hai người)

Nếu chỉ đếm "số quân trên đường" thì đường có `X, O, X` cũng đạt 3 → báo thắng sai.

In [22]:
def wrong_unsigned(moves):
    rows = [0] * 3
    for i, (r, c) in enumerate(moves):
        rows[r] += 1                      # SAI: không phân biệt người
        if rows[r] == 3:
            return "A" if i % 2 == 0 else "B"
    return "Draw" if len(moves) == 9 else "Pending"

mv = [[0, 0], [0, 1], [0, 2]]             # X O X trên hàng 0
print("sai :", wrong_unsigned(mv), "| đúng:", Solution().tictactoe(mv))

sai : A | đúng: Pending


### Lỗi 5: Gán X/O ngược

A **luôn** đi trước và là `'X'` → nước có chỉ số **chẵn**. Viết `i % 2 == 1` cho A là đảo ngược hết kết quả.

## 14. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu:**
- "Kiểm tra có k quân liên tiếp trên hàng/cột/chéo" → **bộ đếm theo đường**: chỉ số hàng `r`, cột `c`, chéo chính `r - c` (hằng), chéo phụ `r + c` (hằng).
- Hai người đối kháng → mã hóa `+1/-1` để dùng **một** bộ đếm.
- Input đảm bảo hợp lệ → khai thác "chỉ người đi cuối có thể thắng".

**Bài liên quan:**
- LeetCode 348 *Design Tic-Tac-Toe* (Premium) — đúng Cách 2 cho bàn `n x n`, mỗi `move` `O(1)`.
- LeetCode 794 *Valid Tic-Tac-Toe State* — kiểm tra bàn cờ có thể đạt được không.
- LeetCode 36 *Valid Sudoku* — đếm theo hàng/cột/ô vuông con.

In [23]:
# Mini: chỉ số đường chéo dùng r - c và r + c (tổng quát cho bàn n x n)
n = 4
print("r - c:")
for r in range(n):
    print("  ", [r - c for c in range(n)])
print("r + c:")
for r in range(n):
    print("  ", [r + c for c in range(n)])
# Chéo chính: r - c == 0; chéo phụ: r + c == n - 1

r - c:
   [0, -1, -2, -3]
   [1, 0, -1, -2]
   [2, 1, 0, -1]
   [3, 2, 1, 0]
r + c:
   [0, 1, 2, 3]
   [1, 2, 3, 4]
   [2, 3, 4, 5]
   [3, 4, 5, 6]


## 15. Bài tập tự luyện

> Gợi ý ở mục 16, đáp án ở mục 17. Không gọi hàm TODO khi chạy notebook.

### Bài 1 — Tính tay
`moves = [[1,1],[0,0],[2,2],[0,2],[0,1],[2,1],[1,0],[1,2],[2,0]]` — ai thắng / hòa / chưa xong? Gán vào `ex1_answer`.

In [24]:
ex1_answer = None   # TODO: "A", "B", "Draw" hoặc "Pending"

# assert ex1_answer == Solution().tictactoe([[1, 1], [0, 0], [2, 2], [0, 2], [0, 1], [2, 1], [1, 0], [1, 2], [2, 0]])

### Bài 2 — Trả về bàn cờ dạng chuỗi
Viết `board_strings(moves)` trả về list 3 chuỗi, mỗi chuỗi 3 ký tự (`'X'`, `'O'`, `' '`).

In [25]:
def board_strings(moves: list[list[int]]) -> list[str]:
    # TODO
    pass

# assert board_strings([[0, 0], [1, 1]]) == ["X  ", " O ", "   "]

### Bài 3 — Bàn `n x n`
Viết `tictactoe_n(n, moves)`: thắng khi chiếm đủ `n` ô trên một hàng/cột/chéo. Dùng bộ đếm.

In [26]:
def tictactoe_n(n: int, moves: list[list[int]]) -> str:
    # TODO
    pass

# assert tictactoe_n(3, [[0, 0], [2, 0], [1, 1], [2, 1], [2, 2]]) == "A"
# assert tictactoe_n(4, [[0, 0], [1, 0], [0, 1], [1, 1], [0, 2], [1, 2], [0, 3]]) == "A"
# assert tictactoe_n(1, [[0, 0]]) == "A"

### Bài 4 — Nước thắng ngay
Viết `winning_moves(moves)` (ván đang `"Pending"`) trả về list (sắp xếp) các ô trống mà **người đến lượt** đặt vào sẽ thắng ngay.

In [27]:
def winning_moves(moves: list[list[int]]) -> list[list[int]]:
    # TODO: thử từng ô trống, gọi lại hàm kiểm tra thắng
    pass

# assert winning_moves([[0, 0], [1, 0], [0, 1], [1, 1]]) == [[0, 2]]
# assert winning_moves([[0, 0], [1, 1]]) == []

### Bài 5 — Mini challenge: bàn cờ có hợp lệ không?
Cho `board` (list 3 chuỗi, như LeetCode 794). Trả `True` nếu bàn có thể đạt được từ một ván đúng luật. Điều kiện: `x == o` hoặc `x == o + 1`; X thắng ⇒ `x == o + 1`; O thắng ⇒ `x == o`; không thể cả hai cùng thắng.

In [28]:
def valid_board(board: list[str]) -> bool:
    # TODO
    pass

# assert valid_board(["O  ", "   ", "   "]) == False
# assert valid_board(["XOX", " X ", "   "]) == False
# assert valid_board(["XOX", "O O", "XOX"]) == True
# assert valid_board(["XXX", "   ", "OOO"]) == False

## 16. Gợi ý 3 mức

**Bài 1:** (1) Vẽ bàn. (2) Nước chẵn là X. (3) Kiểm 8 đường; nếu không ai thắng mà đủ 9 nước → Draw.

**Bài 2:** (1) Dựng `board` như Cách 1. (2) `"".join(row)`. (3) List comprehension trên 3 hàng.

**Bài 3:** (1) Thay mọi số `3` bằng `n`. (2) Chéo phụ: `r + c == n - 1`. (3) Draw khi `len(moves) == n * n`.

**Bài 4:** (1) Người đến lượt = A nếu `len(moves)` chẵn. (2) Ô trống = 9 ô trừ các ô đã đi. (3) Với mỗi ô trống, `Solution().tictactoe(moves + [[r, c]])` có trả đúng người đó không.

**Bài 5:** (1) Đếm `x`, `o`. (2) Viết hàm `win(ch)` kiểm tra 8 đường. (3) Kiểm theo thứ tự các điều kiện trong đề bài tập.

## 17. Đáp án tham khảo

In [29]:
ans_ex1 = "Draw"


def ans_board_strings(moves):
    board = [[" "] * 3 for _ in range(3)]
    for i, (r, c) in enumerate(moves):
        board[r][c] = "X" if i % 2 == 0 else "O"
    return ["".join(row) for row in board]


def ans_tictactoe_n(n, moves):
    rows, cols = [0] * n, [0] * n
    diag = anti = 0
    for i, (r, c) in enumerate(moves):
        v = 1 if i % 2 == 0 else -1
        rows[r] += v
        cols[c] += v
        if r == c:
            diag += v
        if r + c == n - 1:
            anti += v
        if n in (abs(rows[r]), abs(cols[c]), abs(diag), abs(anti)):
            return "A" if v == 1 else "B"
    return "Draw" if len(moves) == n * n else "Pending"


def ans_winning_moves(moves):
    player = "A" if len(moves) % 2 == 0 else "B"
    used = {tuple(m) for m in moves}
    res = []
    for r in range(3):
        for c in range(3):
            if (r, c) not in used and Solution().tictactoe(moves + [[r, c]]) == player:
                res.append([r, c])
    return res


def ans_valid_board(board):
    x = sum(row.count("X") for row in board)
    o = sum(row.count("O") for row in board)
    def win(ch):
        lines = [list(r) for r in board] + [list(c) for c in zip(*board)]
        lines += [[board[i][i] for i in range(3)], [board[i][2 - i] for i in range(3)]]
        return any(line == [ch] * 3 for line in lines)
    if not (x == o or x == o + 1):
        return False
    xw, ow = win("X"), win("O")
    if xw and ow:
        return False
    if xw and x != o + 1:
        return False
    if ow and x != o:
        return False
    return True


assert ans_ex1 == Solution().tictactoe([[1, 1], [0, 0], [2, 2], [0, 2], [0, 1], [2, 1], [1, 0], [1, 2], [2, 0]])
assert ans_board_strings([[0, 0], [1, 1]]) == ["X  ", " O ", "   "]
assert ans_tictactoe_n(3, [[0, 0], [2, 0], [1, 1], [2, 1], [2, 2]]) == "A"
assert ans_tictactoe_n(4, [[0, 0], [1, 0], [0, 1], [1, 1], [0, 2], [1, 2], [0, 3]]) == "A"
assert ans_tictactoe_n(1, [[0, 0]]) == "A"
assert ans_winning_moves([[0, 0], [1, 0], [0, 1], [1, 1]]) == [[0, 2]]
assert ans_winning_moves([[0, 0], [1, 1]]) == []
assert ans_valid_board(["O  ", "   ", "   "]) == False
assert ans_valid_board(["XOX", " X ", "   "]) == False
assert ans_valid_board(["XOX", "O O", "XOX"]) == True
assert ans_valid_board(["XXX", "   ", "OOO"]) == False
# mọi input hợp lệ sinh ở mục 12 đều cho bàn cờ hợp lệ; tictactoe_n(3, ·) khớp Solution
for mv in sample[:3000]:
    assert ans_valid_board(ans_board_strings(mv))
    assert ans_tictactoe_n(3, mv) == Solution().tictactoe(mv)
print("Đáp án bài tập: tất cả assert PASS")

Đáp án bài tập: tất cả assert PASS


Ghi chú bài 1: bàn cuối là
```text
 O | X | O
 X | X | O
 X | O | X
```
Không có đường nào đủ 3 quân giống nhau (kể cả ở các bước giữa), và đã đủ 9 nước → `"Draw"`.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def tictactoe(self, moves):
        n = 3
        rows, cols = [0] * n, [0] * n
        diag = anti = 0
        for i, (r, c) in enumerate(moves):
            v = 1 if i % 2 == 0 else -1
            rows[r] += v; cols[c] += v
            if r == c: diag += v
            if r + c == n - 1: anti += v
            if n in (abs(rows[r]), abs(cols[c]), abs(diag), abs(anti)):
                return "A" if v == 1 else "B"
        return "Draw" if len(moves) == n * n else "Pending"
```

**Câu thần chú:** *"A = +1, B = −1; hàng r, cột c, chéo r==c, chéo r+c==n−1; |tổng| = n là thắng; hết bàn mới Draw."*

**Checklist:**
- [ ] Nước **chẵn** là A (X).
- [ ] Hai `if` riêng cho hai đường chéo (ô trung tâm thuộc cả hai).
- [ ] Kiểm tra **thắng trước**, rồi mới Draw/Pending.
- [ ] `Draw` chỉ khi đủ `9` nước.
- [ ] Trả đúng chuỗi `"A"`, `"B"`, `"Draw"`, `"Pending"` (chú ý chữ hoa đầu).